<a href="https://colab.research.google.com/github/moise97/Extract_-_Structure_Data_from_SDFs_pharmaceutical_documentation/blob/main/Route_Queries_Within_a_Large_Contract_PDF.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q pdfplumber
!pip install -q langchain langchain-community
!pip install -q llama-index
!pip install -q llama-index-embeddings-huggingface
!pip install -q llama-index-llms-huggingface
!pip install -q sentence-transformers
!pip install -q transformers accelerate sentencepiece

print('✅ All packages installed.')

In [ ]:
from google.colab import files
import pdfplumber

print('📂 Upload your PDF...')
uploaded = files.upload()
pdf_filename = list(uploaded.keys())[0]
print(f'✅ Uploaded: {pdf_filename}\n')

# ✏️ Set these to match your document
DOC_TYPE   = 'Packaging Specification'
DOC_ID     = 'packaging_spec_01'

# Extract text page by page with metadata
page_metadata_store = []

with pdfplumber.open(pdf_filename) as pdf:
    for i, page in enumerate(pdf.pages):
        text = (page.extract_text() or '').strip()
        page_metadata_store.append({
            'page':        i,
            'doc_type':    DOC_TYPE,
            'text':        text,
            'doc_id':      DOC_ID,
            'source_file': pdf_filename,
            'page_in_doc': i
        })
        preview = text[:80].replace('\n',' ') if text else '[EMPTY]'
        print(f'  Page {i}: {preview}...')

print(f'\n✅ Extracted {len(page_metadata_store)} page(s).')

In [ ]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

# Concatenate all pages into one logical document
full_doc_text = '\n\n'.join(
    p['text'] for p in page_metadata_store if p['text']
)
print(f'Full document length : {len(full_doc_text)} characters')

# Chunk it
splitter = RecursiveCharacterTextSplitter(
    chunk_size=512,
    chunk_overlap=100
)
chunks = splitter.split_text(full_doc_text)

print(f'Total chunks created : {len(chunks)}')
print(f'\nFirst chunk preview:')
print('-' * 50)
print(chunks[0][:300] if chunks else '[NO CHUNKS]')

In [ ]:
from llama_index.core import Document

all_documents = []

for i, chunk in enumerate(chunks):
    all_documents.append(
        Document(
            text=chunk,
            metadata={
                'doc_type':    DOC_TYPE,
                'chunk_index': i,
                'doc_id':      DOC_ID,
                'source_file': pdf_filename
            }
        )
    )

print(f'✅ Created {len(all_documents)} LlamaIndex Document(s) with metadata.')
print(f'\nSample — chunk 0 metadata:')
print(all_documents[0].metadata)
print(f'\nSample — chunk 0 text preview:')
print(all_documents[0].text[:200])

In [ ]:
from llama_index.llms.huggingface import HuggingFaceLLM
from llama_index.core import Settings
import torch

MODEL_NAME = 'TinyLlama/TinyLlama-1.1B-Chat-v1.0'
print(f'Loading {MODEL_NAME}...\n')

tinyllama_llm = HuggingFaceLLM(
    model_name=MODEL_NAME,
    tokenizer_name=MODEL_NAME,
    context_window=2048,
    max_new_tokens=512,
    generate_kwargs={'temperature': 0.2, 'do_sample': True},
    device_map='auto',
    model_kwargs={'torch_dtype': torch.float16}
)

Settings.llm = tinyllama_llm
print('✅ TinyLlama ready.')
if torch.cuda.is_available():
    print(f'   Device : {torch.cuda.get_device_name(0)}')

In [ ]:
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.core import VectorStoreIndex, Settings

print('Loading BGE embedding model...')
Settings.embed_model = HuggingFaceEmbedding(model_name='BAAI/bge-small-en-v1.5')

print('Building vector index (embedding all chunks)...')
index = VectorStoreIndex.from_documents(
    all_documents,
    show_progress=True
)

print('\n✅ Vector index built.')
print(f'   Chunks indexed: {len(all_documents)}')

In [ ]:
from llama_index.core.vector_stores import (
    MetadataFilters,
    MetadataFilter,
    FilterOperator
)

# ✏️ Change this query to test different inputs
QUERY = 'Were there any packaging configuration changes?'

# Build retriever with metadata filter — only returns chunks
# where doc_type == DOC_TYPE
retriever = index.as_retriever(
    similarity_top_k=4,
    filters=MetadataFilters(filters=[
        MetadataFilter(
            key='doc_type',
            value=DOC_TYPE,
            operator=FilterOperator.EQ
        )
    ])
)

print(f'Query          : "{QUERY}"')
print(f'Filtering by   : doc_type = "{DOC_TYPE}"')
print('Retrieving...\n')

retrieved_nodes = retriever.retrieve(QUERY)

print(f'✅ Retrieved {len(retrieved_nodes)} chunk(s).')
print('\nRetrieved chunks:')
print('-' * 55)
for i, node in enumerate(retrieved_nodes):
    print(f'\nChunk {i+1} (score: {node.score:.3f})')
    print(f'Metadata : {node.metadata}')
    print(f'Text     : {node.get_text()[:200]}...')

In [ ]:
from llama_index.core.response_synthesizers import (
    get_response_synthesizer,
    ResponseMode
)

synthesizer = get_response_synthesizer(
    response_mode=ResponseMode.COMPACT
)

print('Synthesizing answer...\n')
response = synthesizer.synthesize(QUERY, nodes=retrieved_nodes)

print('=' * 55)
print('Answer:')
print(response.response)
print('=' * 55)

In [ ]:
import json
from google.colab import files

# Build matched_chunks list from retrieved nodes
matched_chunks = [
    {
        'text':     node.get_text(),
        'metadata': node.metadata,
        'score':    round(node.score, 4) if node.score else None
    }
    for node in retrieved_nodes
]

# Final output dictionary
final_output = {
    'query':              QUERY,
    'predicted_doc_type': DOC_TYPE,
    'matched_chunks':     matched_chunks,
    'answer':             response.response
}

# Print it
json_output = json.dumps(final_output, indent=2)
print('Final Output:')
print('=' * 60)
print(json_output)
print('=' * 60)

# Save + download
with open('rag_final_output.json', 'w') as f:
    f.write(json_output)

print('\n✅ Saved as rag_final_output.json')
files.download('rag_final_output.json')

In [ ]:
import pandas as pd

rows = []
for node in retrieved_nodes:
    rows.append({
        'chunk_index': node.metadata.get('chunk_index'),
        'doc_type':    node.metadata.get('doc_type'),
        'score':       round(node.score, 4) if node.score else None,
        'text_preview': node.get_text()[:100] + '...'
    })

df = pd.DataFrame(rows)
print(f'Query: "{QUERY}"')
print(df.to_string(index=False))